# 👁️ Pupil-Limbus Segmentation Model Training on Google Colab 🚀

Welcome to the Google Colab training notebook for the Pupil-Limbus eye tracker!  
By using Google Colab's high-performance cloud GPUs (such as the NVIDIA T4, L4, or A100), you can train your segmentation models **up to 20x faster** than on a standard laptop CPU or low-end GPU.

### 📋 Workflow Overview:
1. ✅ Verify GPU is active
2. 📦 Clone the project code from GitHub *(no zip uploads needed!)*
3. 📁 Upload your `clinical_data.zip` from Google Drive or directly
4. 🛠️ Install dependencies
5. 🏋️ Run training
6. 💾 Export ONNX and download your trained model

In [ ]:
# @title 1. Verify GPU Acceleration 🏎️
# Ensure a high-performance GPU is allocated to your Colab session.
# If no GPU is found, go to: Runtime → Change runtime type → GPU (T4 is free) → Save
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_prop = torch.cuda.get_device_properties(0)
    print(f"✅ GPU active: {device_name}")
    print(f"   Total GPU memory: {device_prop.total_memory / (1024**3):.2f} GB")
    print(f"   CUDA version:     {torch.version.cuda}")
else:
    print("❌ GPU is NOT active!")
    print("   Go to: Runtime → Change runtime type → GPU (T4 GPU is free) → Save")
    print("   Then restart the runtime and run all cells again.")

## 2. Clone Project Code from GitHub 📦

This pulls the **latest committed code** directly from GitHub — no ZIP files or nested folder issues!

In [ ]:
# @title 2a. Clone / Update Project from GitHub 🔄
import os
import shutil
from pathlib import Path

WORKSPACE = Path("/content/workspace")
REPO_URL = "https://github.com/Prince649294u83/Centration.git"

if WORKSPACE.exists() and (WORKSPACE / ".git").exists():
    print("🔄 Workspace already cloned — pulling latest changes...")
    !git -C /content/workspace pull
else:
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    print(f"📥 Cloning from {REPO_URL} ...")
    !git clone {REPO_URL} /content/workspace

# Verify critical files exist
ok = True
for f in ["scripts/train_model.py", "scripts/export_onnx.py", "pupil_tracking"]:
    path = WORKSPACE / f
    if path.exists():
        print(f"   ✅ {f}")
    else:
        print(f"   ❌ MISSING: {f}")
        ok = False

if ok:
    print("\n✅ Project code is ready!")
else:
    print("\n❌ Some files are missing. Check the repository URL and try again.")

## 3. Connect Your Clinical Data 📁

Your `clinical_data` folder contains annotated training images and masks.  
Choose **one** of the options below:

- **Option A** *(Recommended)*: Mount Google Drive — upload `clinical_data.zip` once to Drive, reuse across sessions.
- **Option B**: Upload `clinical_data.zip` directly to Colab each session.

In [ ]:
# @title Option A: Mount Google Drive & Copy Clinical Data 📁 (Recommended)
# 1. Upload 'clinical_data.zip' to the root of your Google Drive.
# 2. Run this cell.
from google.colab import drive
import shutil
from pathlib import Path

try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")

    drive_zip = Path('/content/drive/MyDrive/clinical_data.zip')
    local_zip = Path('/content/clinical_data.zip')

    if drive_zip.exists():
        print("📋 Copying clinical_data.zip from Drive...")
        shutil.copy2(drive_zip, local_zip)
        print(f"   ✅ Copied! ({local_zip.stat().st_size / 1024**2:.1f} MB)")
    else:
        print("ℹ️  'clinical_data.zip' not found in Google Drive root.")
        print("   Please upload it to: My Drive/clinical_data.zip")

except Exception as e:
    print(f"⚠️  Google Drive mount failed: {e}")
    print("   Use Option B (direct upload) instead.")

In [ ]:
# @title Option B: Upload clinical_data.zip Directly 📤 (Alternative)
# Click 'Choose Files', select your local clinical_data.zip, and wait for the upload.
from google.colab import files
print("📤 Select your clinical_data.zip file to upload...")
uploaded = files.upload()
for name, data in uploaded.items():
    dest = f"/content/{name}"
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"   ✅ Uploaded: {dest} ({len(data)/1024**2:.1f} MB)")

In [ ]:
# @title Extract Clinical Data into Workspace 🔓
# Run this AFTER choosing Option A or B above.
import zipfile
import shutil
from pathlib import Path

WORKSPACE = Path("/content/workspace")
data_zip = Path("/content/clinical_data.zip")
data_dest = WORKSPACE / "clinical_data"

if not data_zip.exists():
    print("❌ 'clinical_data.zip' not found at /content/clinical_data.zip")
    print("   Please run Option A or Option B above first.")
else:
    # Remove old clinical_data if present
    if data_dest.exists():
        print("🗑️  Removing old clinical_data folder...")
        shutil.rmtree(data_dest)

    print(f"📦 Extracting clinical_data.zip ({data_zip.stat().st_size/1024**2:.1f} MB)...")
    with zipfile.ZipFile(data_zip, 'r') as zf:
        # Check if zip contains a single root folder (common nesting issue)
        members = zf.namelist()
        root_dirs = set(m.split('/')[0] for m in members if m.split('/')[0])
        
        if len(root_dirs) == 1 and list(root_dirs)[0] == 'clinical_data':
            # Zip already has clinical_data/ as root — extract directly into workspace
            zf.extractall(WORKSPACE)
            print("   ✅ Extracted (detected 'clinical_data/' root in zip)")
        else:
            # No clinical_data prefix — extract into clinical_data/ subfolder
            zf.extractall(data_dest)
            print("   ✅ Extracted into workspace/clinical_data/")

    # Verify structure
    print("\n📂 Checking data structure...")
    checks = {
        "clinical_data/": data_dest,
        "clinical_data/training_data/images/": data_dest / "training_data" / "images",
        "clinical_data/training_data/masks/": data_dest / "training_data" / "masks",
        "clinical_data/annotations/annotations.json": data_dest / "annotations" / "annotations.json",
    }
    all_ok = True
    for label, path in checks.items():
        if path.exists():
            print(f"   ✅ {label}")
        else:
            print(f"   ❌ MISSING: {label}")
            all_ok = False

    if all_ok:
        # Count images
        img_dir = data_dest / "training_data" / "images"
        n_images = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        print(f"\n✅ Clinical data ready! Found {n_images} training images.")
    else:
        print("\n⚠️  Some expected folders are missing. Check your clinical_data.zip structure.")

## 4. Install Dependencies 🛠️

In [ ]:
# @title Install Core Libraries & Dependencies 🚀
# Installs segmentation models, augmentation library, and ONNX runtime.
# This takes ~60 seconds on first run.
%cd /content/workspace

print("📦 Installing dependencies...")
!pip install -q \
    segmentation-models-pytorch \
    albumentations==1.4.3 \
    onnxruntime-gpu \
    timm

# Verify all critical imports
print("\n🔍 Verifying imports...")
import sys
sys.path.insert(0, '/content/workspace')

import numpy as np
print(f"   ✅ numpy {np.__version__}")

import cv2
print(f"   ✅ opencv {cv2.__version__}")

import albumentations as A
print(f"   ✅ albumentations {A.__version__}")

import segmentation_models_pytorch as smp
print(f"   ✅ segmentation-models-pytorch {smp.__version__}")

import torch
print(f"   ✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})")

from pupil_tracking.ml.trainer import Trainer
print(f"   ✅ pupil_tracking.ml.trainer")

print("\n✅ All packages imported and ready!")

## 5. (Optional) Curate Hard Frames 🎯

If you have raw videos with challenging frames (red light reflections, blinks, poor detection), run this to automatically extract the most informative frames for annotation.  
**Skip this step** if you've already curated and annotated your training data.

In [ ]:
# @title Curate Hard Frames from Video (Optional) 🎬
# Set VIDEO_PATH to your uploaded video file.
# Curated frames will be saved to clinical_data/curated_frames/

VIDEO_PATH = "/content/your_video.mp4"  # ← Change this to your video path
OUTPUT_DIR = "/content/workspace/clinical_data/curated_frames"

from pathlib import Path
if not Path(VIDEO_PATH).exists():
    print(f"ℹ️  Video not found at: {VIDEO_PATH}")
    print("   Upload your video and update VIDEO_PATH above, then re-run this cell.")
    print("   Skipping curation — proceeding with existing training data.")
else:
    print(f"🎬 Curating hard frames from: {VIDEO_PATH}")
    !python scripts/curate_hard_frames.py \
        --video "{VIDEO_PATH}" \
        --output "{OUTPUT_DIR}" \
        --max-frames 800
    
    n = len(list(Path(OUTPUT_DIR).glob("*.jpg"))) + len(list(Path(OUTPUT_DIR).glob("*.png")))
    print(f"\n✅ Curated {n} hard frames → {OUTPUT_DIR}")
    print("   Next: annotate these frames using the annotation tool, then re-run training.")

## 6. Run Training Pipeline 🏋️

The production-grade training loop includes:
- ⚡ **Mixed-precision training (AMP)** — doubles GPU throughput
- 📉 **Cosine annealing LR schedule** — finds optimal minima
- 🛑 **Early stopping** — prevents overfitting
- ⚖️ **Inverse-frequency weighted loss** — handles class imbalance
- 🔄 **50 augmented copies per image** — maximises data diversity

**Expected training time on T4 GPU**: ~15–30 minutes for 300 epochs

In [ ]:
# @title Start GPU-Accelerated Training 🚀
# ─────────────────────────────────────────────────────
# Adjust as needed:
#   --num-classes 3  →  3-class model (background / pupil / iris)
#   --num-classes 4  →  4-class model (adds suction_ring class)
#   --epochs 300     →  300 epochs gives solid production accuracy
#   --batch-size 16  →  fits comfortably in 14 GB T4 VRAM
# ─────────────────────────────────────────────────────

!python scripts/train_model.py \
    --epochs 300 \
    --batch-size 16 \
    --lr 0.0005 \
    --num-classes 3 \
    --annotation-path clinical_data/annotations/annotations.json \
    --image-dir clinical_data/training_data/images \
    --mask-dir clinical_data/training_data/masks \
    --save-dir models \
    --device cuda

## 7. Export to ONNX & Download 📤

Convert your trained PyTorch model (`.pth`) to an optimised ONNX model (`.onnx`).  
ONNX models run in a **50 MB runtime with no PyTorch dependency** — perfect for the local tracking app.

In [ ]:
# @title Export Checkpoint to ONNX 🌟
from pathlib import Path

MODEL_PATH = "/content/workspace/models/best_model.pth"
ONNX_OUTPUT = "/content/workspace/models/onnx/segmentation.onnx"

if not Path(MODEL_PATH).exists():
    print(f"❌ Model not found: {MODEL_PATH}")
    print("   Make sure training completed successfully in step 6.")
else:
    Path(ONNX_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    print(f"🔄 Converting {MODEL_PATH} → ONNX...")
    !python scripts/export_onnx.py \
        --model "{MODEL_PATH}" \
        --output "{ONNX_OUTPUT}"
    
    if Path(ONNX_OUTPUT).exists():
        size_mb = Path(ONNX_OUTPUT).stat().st_size / 1024**2
        print(f"\n✅ ONNX model saved: {ONNX_OUTPUT} ({size_mb:.1f} MB)")
    else:
        print("❌ ONNX export failed. Check the error above.")

In [ ]:
# @title Save Trained Models back to Google Drive 💾 (Recommended)
# Saves both .pth and .onnx to Google Drive so they persist after the session ends.
import shutil
from pathlib import Path

DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/pupil_limbus_models")
MODELS_DIR = Path("/content/workspace/models")

if not Path("/content/drive").exists():
    print("ℹ️  Google Drive not mounted. Run the 'Mount Google Drive' cell first.")
elif not MODELS_DIR.exists():
    print("❌ Models directory not found. Make sure training completed.")
else:
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"💾 Copying models to Drive: {DRIVE_SAVE_DIR}")
    
    copied = 0
    for f in MODELS_DIR.rglob("*"):
        if f.is_file() and f.suffix in (".pth", ".onnx"):
            dest = DRIVE_SAVE_DIR / f.relative_to(MODELS_DIR)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            print(f"   ✅ {f.name} ({f.stat().st_size/1024**2:.1f} MB)")
            copied += 1
    
    print(f"\n✅ {copied} model file(s) saved to Google Drive → {DRIVE_SAVE_DIR}")
    print("   Download from Drive to your local project's 'models/' folder.")

In [ ]:
# @title (Alternative) Direct Download as ZIP 💾
# If Drive is not mounted, download models directly to your computer.
import shutil
from google.colab import files
from pathlib import Path

models_dir = Path("/content/workspace/models")
archive_path = "/content/trained_models_colab"

if not models_dir.exists():
    print("❌ Models folder not found! Ensure training completed successfully.")
else:
    print("📦 Packaging models into ZIP archive...")
    shutil.make_archive(archive_path, 'zip', str(models_dir))
    zip_size = Path(f"{archive_path}.zip").stat().st_size / 1024**2
    print(f"✅ Archive ready: {archive_path}.zip ({zip_size:.1f} MB)")

    try:
        files.download(f'{archive_path}.zip')
        print("⬇️  Download started!")
        print("   Extract the ZIP contents into your local project 'models/' folder.")
    except Exception as e:
        print(f"\n⚠️  Auto-download failed: {e}")
        print(f"   Manually download from the left sidebar: {archive_path}.zip")